In [8]:
# 1. IMPORT DEPENDENCIES
import pandas as pd
import numpy as np
import glob
from scipy.optimize import minimize_scalar
import warnings
import os
import urllib.request

warnings.filterwarnings('ignore')
print("System Fund: Quant Engine Initialized.")

# --- AUTO-DOWNLOADER FOR VISITORS ---
csv_filename = "Systemizing_Chaos_5_Year_Track_Record.csv"
github_url = "https://raw.githubusercontent.com/bensullivann/SystemFund-Tradelog/refs/heads/main/data/Systemizing_Chaos_5_Year_Track_Record.csv"

if not os.path.exists(csv_filename):
    print("Fetching trade data from GitHub...")
    try:
        urllib.request.urlretrieve(github_url, csv_filename)
        print("✅ Download complete.")
    except Exception as e:
        print(f"❌ Failed to download data. Error: {e}")
# -----------------------------------------

# 2. INGEST RAW MARKET DATA
files = glob.glob("*.csv")
all_trades = []
portfolio = pd.DataFrame()

if not files:
    print("❌ Error: No CSV files found.")
else:
    print(f"✅ Data Source Ready: {files[0]}")
    df = pd.read_csv(files[0])

    # --- DATA CLEANING ---
    # Strip out any '%' or ',' characters from the P&L column and convert to pure numbers
    if 'Raw Trade P&L (%)' in df.columns:
        df['Raw Trade P&L (%)'] = df['Raw Trade P&L (%)'].astype(str).str.replace('%', '').str.replace(',', '').astype(float)

    # --- UPDATED COLUMN MAPPING ---
    if 'Asset' in df.columns:
        df['Bot_Type'] = np.where(df['Asset'].str.contains('BTC', case=False, na=False), 'BTC_Long', 'Short')
    else:
        df['Bot_Type'] = 'Short'

    # Since this is a clean Track Record file, every row is a trade. No need to filter by 'Type'.
    exits = df.copy()

    # Use the new Date column name
    exits['Date & Time'] = pd.to_datetime(exits['Date & Time'])
    all_trades.append(exits)

    # Build the timeline
    if all_trades:
        portfolio = pd.concat(all_trades, ignore_index=True)
        start_date = pd.Timestamp('2021-03-15')
        portfolio = portfolio[portfolio['Date & Time'] >= start_date].copy()
        portfolio = portfolio.sort_values('Date & Time').reset_index(drop=True)

# 3. DEFINE THE DECOUPLED RISK SIMULATOR
def simulate_system(short_risk_pct):
    if portfolio.empty:
        return 0.0, 10000.0

    equity = 10000.0
    peak_equity = 10000.0
    max_dd = 0.0

    for _, row in portfolio.iterrows():
        # Apply the Decoupled Risk Blueprint
        if row['Bot_Type'] == 'BTC_Long':
            actual_risk_multiplier = 0.03 / 0.10  # Hard-capped 3.00% Risk for BTC Long
        else:
            actual_risk_multiplier = short_risk_pct / 0.10 # Dynamic Risk for Shorts

        # Calculate return using the cleaned pure numbers
        trade_return = (row['Raw Trade P&L (%)'] / 100.0) * actual_risk_multiplier
        equity *= (1 + trade_return)

        if equity > peak_equity:
            peak_equity = equity

        drawdown = (peak_equity - equity) / peak_equity
        if drawdown > max_dd:
            max_dd = drawdown

    return max_dd, equity

# 4. OPTIMIZE FOR 10% DRAWDOWN LIMIT
def objective(short_risk):
    dd, _ = simulate_system(short_risk)
    return abs(dd - 0.10) # Targets exactly 10.00% Max DD

# 5. RUN FINAL AUDIT
if not portfolio.empty:
    print("\nOptimizing portfolio risk matrix...")
    result = minimize_scalar(objective, bounds=(0.001, 0.10), method='bounded')
    optimal_short_risk = result.x

    final_max_dd, final_equity = simulate_system(optimal_short_risk)

    btc_count = len(portfolio[portfolio['Bot_Type'] == 'BTC_Long'])
    short_count = len(portfolio[portfolio['Bot_Type'] == 'Short'])

    print("-" * 50)
    print("SYSTEM FUND: 5-YEAR BACKTEST RESULTS")
    print("-" * 50)
    print(f"Total Trades Processed : {len(portfolio):,}")
    print(f"Detected BTC Longs     : {btc_count}")
    print(f"Detected Shorts        : {short_count}")
    print(f"BTC Long Risk          : 3.00%")
    print(f"Optimized Short Risk   : {optimal_short_risk*100:.2f}%")
    print(f"Maximum Drawdown       : {final_max_dd*100:.2f}%")
    print(f"Starting Capital       : $10,000.00")
    print(f"Ending Equity          : ${final_equity:,.2f}")
    print(f"Total Net Return       : {((final_equity/10000)-1)*100:,.2f}%")
    print("-" * 50)

System Fund: Quant Engine Initialized.
✅ Data Source Ready: Systemizing_Chaos_5_Year_Track_Record.csv

Optimizing portfolio risk matrix...
--------------------------------------------------
SYSTEM FUND: 5-YEAR BACKTEST RESULTS
--------------------------------------------------
Total Trades Processed : 2,654
Detected BTC Longs     : 37
Detected Shorts        : 2617
BTC Long Risk          : 3.00%
Optimized Short Risk   : 2.88%
Maximum Drawdown       : 10.00%
Starting Capital       : $10,000.00
Ending Equity          : $394,513.58
Total Net Return       : 3,845.14%
--------------------------------------------------
